# 08 - Arquitetura final híbrida e teste cego de 2025

> **NÃO EXECUTE ESTE NOTEBOOK ANTES DE CONCLUIR 07c E REVISAR `models/architecture_candidate_v7.json`.**

Esta versão congela a arquitetura candidata v7:

- alvo único: **nível da estação 413**;
- previsão de magnitude normal: **XGBoost + radar**;
- quando a TCN identifica regime de subida forte: magnitude = **média(TCN, GRU)**;
- sinais de severidade `>=1 m`, `>=2 m`, `>=3 m`: **heads da TCN**;
- persistência operacional: número de timestamps consecutivos definido no 07c.

Os limiares de 1/2/3 m são **faixas de subida**, não cotas oficiais.

In [ ]:
# ============================================================
# TRAVA METODOLÓGICA
# ============================================================
OPEN_BLIND_2025 = False

if not OPEN_BLIND_2025:
    raise RuntimeError(
        "2025 continua bloqueado. Revise o 07c, confira architecture_candidate_v7.json "
        "e só então troque OPEN_BLIND_2025 para True. Depois disso 2025 deixa de ser cego."
    )

In [ ]:
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")
CWD = Path.cwd().resolve()
ROOT = CWD.parent if CWD.name.lower() == "notebooks" else CWD
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"
OUTPUTS = ROOT / "outputs"
TABLES = OUTPUTS / "tables"
PREDICTIONS = OUTPUTS / "predictions"
FIGURES = OUTPUTS / "figures"
for p in [MODELS,TABLES,PREDICTIONS,FIGURES]: p.mkdir(parents=True,exist_ok=True)

ARCH_PATH = MODELS / "architecture_candidate_v7.json"
if not ARCH_PATH.exists(): raise FileNotFoundError("Execute 07c primeiro: architecture_candidate_v7.json não existe.")
architecture = json.loads(ARCH_PATH.read_text(encoding="utf-8"))
if architecture.get("target_station") != 413: raise RuntimeError("Esta versão foi desenhada somente para a estação 413.")
print(json.dumps(architecture, indent=2, ensure_ascii=False))

## 1. Treino final do XGBoost + radar

O número de árvores continua congelado a partir da etapa de desenvolvimento. Reajustamos o modelo em todos os dados disponíveis **antes de 2025** (`train + validation + test_2024`) e só depois calculamos previsões para 2025.

In [ ]:
from xgboost import XGBRegressor

radar_path = PROCESSED / "features_causal_radar.parquet"
if not radar_path.exists(): raise FileNotFoundError("Rode o notebook 06 primeiro.")
df = pd.read_parquet(radar_path).sort_index()
TARGET = "target_delta_120m"
feature_cols = [c for c in df.columns if c != "split" and not c.startswith("target_")]
fit = df[df["split"].isin(["train","validation","test_2024"])].dropna(subset=["stage_now", TARGET])
blind = df[df["split"] == "blind_2025"].dropna(subset=["stage_now", TARGET])

dev = XGBRegressor(); dev.load_model(MODELS / "xgb_radar_dev_120m.json")
try:
    n_trees = int(dev.get_booster().attr("best_iteration")) + 1
except Exception:
    try: n_trees = int(dev.best_iteration) + 1
    except Exception: n_trees = 600
print("Árvores congeladas:", n_trees)

xgb_final = XGBRegressor(
    objective="reg:squarederror", eval_metric="rmse", tree_method="hist",
    n_estimators=n_trees, learning_rate=.035, max_depth=6, min_child_weight=20,
    subsample=.85, colsample_bytree=.80, reg_lambda=5.0, max_bin=128,
    n_jobs=-1, random_state=42,
)
xgb_final.fit(fit[feature_cols], fit[TARGET])
xgb_pred = xgb_final.predict(blind[feature_cols])
xgb_final.save_model(MODELS / "final_xgboost_radar_120m_v7.json")
xgb_blind = pd.DataFrame({
    "prediction_time": blind.index,
    "stage_now": blind["stage_now"].to_numpy(),
    "target_delta_120m": blind[TARGET].to_numpy(),
    "pred_xgb_radar": xgb_pred,
})
print("XGBoost final concluído:", len(xgb_blind), "previsões")

## 2. Treino final da TCN e da GRU

As duas redes usam exatamente a mesma representação temporal do 07b. O pré-processador é reajustado com dados pré-2025. O número de épocas é **fixado pelo melhor epoch observado antes de 2025**; não usamos o holdout para early stopping.

In [ ]:
import torch
from torch.utils.data import DataLoader
from utils.temporal_models import (
    TemporalPreprocessor, SequenceDataset, TCNMultiTask, GRUMultiTask, TrainingConfig,
    build_temporal_frame, valid_sequence_indices, fit_fixed_epochs, predict_model, choose_device,
)

master = pd.read_parquet(PROCESSED / "master_base.parquet").sort_index()
features = pd.read_parquet(PROCESSED / "features_causal.parquet").sort_index()
radar = pd.read_parquet(PROCESSED / "radar_features_basic.parquet").sort_index()
if radar.index.has_duplicates: radar = radar.groupby(level=0).mean(numeric_only=True).sort_index()
raw, groups = build_temporal_frame(master, radar=radar)
target = features[TARGET].reindex(raw.index)
split = features["split"].reindex(raw.index)

registry = json.loads((MODELS / "neural_registry_v6.json").read_text(encoding="utf-8"))
lookback = int(registry["lookback_steps"])
fit_mask = split.isin(["train","validation","test_2024"]) & target.notna()
pre = TemporalPreprocessor(groups, stage_ffill_limit=6).fit(raw, fit_mask, target)
scaled = pre.transform(raw)
matrix = scaled.to_numpy("float32")
yraw = target.to_numpy("float32")
yscaled = pre.transform_target(yraw)
pre.state.to_json(MODELS / "final_temporal_preprocessor_v7.json")

split_final = split.copy()
split_final.loc[split_final.isin(["train","validation","test_2024"])] = "fit_final"
idx_fit = valid_sequence_indices(raw.index, split_final, target, "fit_final", lookback, stride=2)
idx_blind = valid_sequence_indices(raw.index, split_final, target, "blind_2025", lookback, stride=1)
fit_ds = SequenceDataset(matrix, yscaled, yraw, idx_fit, lookback)
blind_ds = SequenceDataset(matrix, yscaled, yraw, idx_blind, lookback)
fit_loader = DataLoader(fit_ds, batch_size=512, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
blind_loader = DataLoader(blind_ds, batch_size=512, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

device = choose_device(); input_dim = matrix.shape[1]
print("Device:", device, "| input_dim:", input_dim, "| treino:", len(idx_fit), "| blind:", len(idx_blind))

In [ ]:
def best_epoch(model_name):
    p = TABLES / f"{model_name.lower()}_training_history_v6.csv"
    h = pd.read_csv(p)
    return int(h.loc[h["valid_loss"].idxmin(), "epoch"])

# TCN
tcn_epochs = best_epoch("TCN")
tcn_final = TCNMultiTask(input_dim, channels=(64,64,96,96), kernel_size=3, dropout=.15)
tcn_cfg = TrainingConfig(model_name="TCN", lookback_steps=lookback, batch_size=512, max_epochs=tcn_epochs, patience=0, seed=42)
fit_fixed_epochs(tcn_final, fit_loader, yraw[idx_fit], tcn_cfg, tcn_epochs, MODELS / "final_tcn_120m_v7.pt", device=device)
tcn_pred = predict_model(tcn_final, blind_loader, pre, raw.index, device=device)

# GRU
gru_epochs = best_epoch("GRU")
gru_final = GRUMultiTask(input_dim, hidden_dim=96, num_layers=2, dropout=.15)
gru_cfg = TrainingConfig(model_name="GRU", lookback_steps=lookback, batch_size=512, max_epochs=gru_epochs, patience=0, seed=42)
fit_fixed_epochs(gru_final, fit_loader, yraw[idx_fit], gru_cfg, gru_epochs, MODELS / "final_gru_120m_v7.pt", device=device)
gru_pred = predict_model(gru_final, blind_loader, pre, raw.index, device=device)

print("Épocas fixas TCN/GRU:", tcn_epochs, gru_epochs)

## 3. Aplicando a regra híbrida congelada

A decisão é causal e foi definida antes de olhar 2025:

- se `TCN p(ΔH>=1m)` ficar abaixo do limiar → magnitude do XGBoost+radar;
- acima do limiar → média das magnitudes TCN e GRU.

O valor real de 2025 **não participa** dessa escolha.

In [ ]:
regime_threshold = float(architecture["point_forecast"]["regime_probability_threshold"])
alert_thresholds = {float(k): float(v) for k,v in architecture["severity_signals"]["thresholds_m"].items()}
persistence_steps = int(architecture["severity_signals"]["persistence_steps"])

tcn_blind = tcn_pred.copy()
gru_blind = gru_pred[["prediction_time","pred_delta_120m"]].rename(columns={"pred_delta_120m":"pred_gru"})
combo = tcn_blind.rename(columns={"pred_delta_120m":"pred_tcn"}).merge(gru_blind, on="prediction_time", how="inner")
combo = combo.merge(xgb_blind[["prediction_time","stage_now","pred_xgb_radar"]], on="prediction_time", how="inner")
combo["risk_regime"] = combo["p_ge_1m"] >= regime_threshold
combo["pred_neural_mean"] = (combo["pred_tcn"] + combo["pred_gru"]) / 2
combo["pred_delta_120m"] = np.where(combo["risk_regime"], combo["pred_neural_mean"], combo["pred_xgb_radar"])
combo["model_used"] = np.where(combo["risk_regime"], "TCN+GRU", "XGBoost+radar")
combo["target_time"] = pd.to_datetime(combo["prediction_time"]) + pd.Timedelta(minutes=120)
combo["real_future_stage"] = combo["stage_now"] + combo["target_delta_120m"]
combo["pred_future_stage"] = combo["stage_now"] + combo["pred_delta_120m"]
combo["error_cm"] = 100*(combo["pred_delta_120m"]-combo["target_delta_120m"])
combo.to_csv(PREDICTIONS / "blind_2025_predictions_hybrid_v7.csv", index=False)
print("Fração de timestamps em regime neural:", f"{100*combo.risk_regime.mean():.2f}%")

## 4. Resultado cego: magnitude

A partir desta célula, 2025 deixou de ser desconhecido. Os resultados são **avaliação**, não novo material para escolher hiperparâmetros.

In [ ]:
def reg_metrics(name, y, p):
    e=np.asarray(p)-np.asarray(y)
    return {
        "model":name, "n":len(e),
        "MAE_cm":100*np.mean(np.abs(e)),
        "RMSE_cm":100*np.sqrt(np.mean(e**2)),
        "bias_cm":100*np.mean(e),
        "median_abs_error_cm":100*np.median(np.abs(e)),
        "max_abs_error_cm":100*np.max(np.abs(e)),
        "within_20cm_pct":100*np.mean(np.abs(e)<=.2),
        "within_50cm_pct":100*np.mean(np.abs(e)<=.5),
    }

summary = pd.DataFrame([
    reg_metrics("XGBoost + radar", combo.target_delta_120m, combo.pred_xgb_radar),
    reg_metrics("TCN", combo.target_delta_120m, combo.pred_tcn),
    reg_metrics("GRU", combo.target_delta_120m, combo.pred_gru),
    reg_metrics("Hibrido v7", combo.target_delta_120m, combo.pred_delta_120m),
])
summary.to_csv(TABLES / "blind_2025_point_metrics_v7.csv", index=False)
display(summary)

bins=[-np.inf,0,.25,.5,1,2,3,np.inf]; labels=["<=0","0-0.25","0.25-0.5","0.5-1","1-2","2-3",">3"]
z=combo.copy(); z["severity_band"]=pd.cut(z.target_delta_120m,bins=bins,labels=labels)
rows=[]
for band in labels:
    q=z[z.severity_band==band]
    if not len(q): continue
    for name,col in [("XGBoost + radar","pred_xgb_radar"),("TCN","pred_tcn"),("GRU","pred_gru"),("Hibrido v7","pred_delta_120m")]:
        e=q[col]-q.target_delta_120m
        rows.append({"severity_band":band,"model":name,"n":len(q),"MAE_cm":100*e.abs().mean(),"RMSE_cm":100*np.sqrt(np.mean(e**2)),"bias_cm":100*e.mean(),"underprediction_pct":100*np.mean(e<0)})
sev=pd.DataFrame(rows); sev.to_csv(TABLES / "blind_2025_severity_metrics_v7.csv",index=False); display(sev)

## 5. Resultado cego: sinais de severidade e episódios

Usamos a TCN e os thresholds congelados. Mostramos tanto a ativação bruta quanto a versão com persistência escolhida no 07c. A avaliação por episódio usa a mesma definição de 60 min de separação e o mesmo pico de referência.

In [ ]:
from utils.episode_evaluation import (
    EpisodeConfig, build_base_episode_catalog, point_classification_metrics, evaluate_episode_alerts
)
EP_CONFIG = EpisodeConfig(base_threshold_m=1.0,gap_minutes=60,alert_gap_minutes=30,horizon_minutes=120,expected_step_minutes=10)
base_ep = build_base_episode_catalog(combo, config=EP_CONFIG)
point_rows=[]; episode_rows=[]
for sev_th,col in [(1.0,"p_ge_1m"),(2.0,"p_ge_2m"),(3.0,"p_ge_3m")]:
    for persistence in [1,persistence_steps]:
        pm=point_classification_metrics(combo,col,alert_thresholds[sev_th],sev_th,min_consecutive=persistence)
        point_rows.append(pm)
        _,_,em=evaluate_episode_alerts(combo,col,alert_thresholds[sev_th],sev_th,config=EP_CONFIG,min_consecutive=persistence,base_catalog=base_ep)
        episode_rows.append(em)
blind_alert_point=pd.DataFrame(point_rows); blind_alert_episode=pd.DataFrame(episode_rows)
blind_alert_point.to_csv(TABLES / "blind_2025_alert_point_metrics_v7.csv",index=False)
blind_alert_episode.to_csv(TABLES / "blind_2025_alert_episode_metrics_v7.csv",index=False)
display(blind_alert_point)
display(blind_alert_episode)

## 6. Metadados para o notebook 09

Depois deste notebook, os arquivos finais podem ser usados para hindcast ou operação experimental. O notebook 09 não deve reinterpretar thresholds nem escolher outra arquitetura.

In [ ]:
metadata = {
    "version":"v7",
    "target_station":413,
    "horizon_min":120,
    "architecture":"hybrid_by_regime",
    "xgb_model":"models/final_xgboost_radar_120m_v7.json",
    "tcn_checkpoint":"models/final_tcn_120m_v7.pt",
    "gru_checkpoint":"models/final_gru_120m_v7.pt",
    "temporal_preprocessor":"models/final_temporal_preprocessor_v7.json",
    "lookback_steps":lookback,
    "regime_probability_threshold":regime_threshold,
    "alert_thresholds":{str(k):v for k,v in alert_thresholds.items()},
    "alert_persistence_steps":persistence_steps,
    "official_stage_thresholds_used":False,
    "blind_test_opened":True,
}
(MODELS / "final_model_metadata_v7.json").write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding="utf-8")
print("Salvo:", MODELS / "final_model_metadata_v7.json")

## Regra depois do teste cego

Não altere a arquitetura e continue chamando o mesmo 2025 de “teste cego”. Se uma nova ideia surgir após ver 2025, ela pertence a uma **nova iteração de pesquisa** e precisará de outro esquema de validação.